# Seminar 7

In [ ]:
!pip install --quiet datasets evaluate transformers
!pip install --quiet deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.0/563.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.0 MB/s eta 0:00:00
  

## Perplexity

$PP(T) = \sqrt[N]{\prod_{i=1}^N\frac{1}{P(w_i|w_1, ..., w_{i-1})}}$

In [ ]:
#Crime and punishment again
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline
import re
import numpy as np



book = open('crime_and_punishment.txt').read()
book = '***'.join(book.split('***')[2:-2])

text =  re.findall('[A-Za-z]+',
                    book)

train_data, padded_sents = padded_everygram_pipeline(2, [text])

model = MLE(2)
model.fit(train_data, text)

In [ ]:
def score_bigram_model(model, T, verbose=False):
  score = model.score(T[0])
  for i in range(1, len(T)):
     #P(w_i|w_{i-1})
     score *= model.score(T[i], T[i-1:i])
     if verbose: print(i, score)

     #If \prod P(w_i|w_{i-1}) = 0, perplexity is infinite
     if score == 0.0: return np.inf
  return score ** (-1/len(T))

In [ ]:
score_bigram_model(model, [
    'Rodion', 'Romanovitch'
])

49.30010141977397

In [ ]:
score_bigram_model(model, [
    'Rodion', 'Romanovitch', 'Raskolnikov','had', 'been', 'a', 'great', 'man', 'who', 'killed', 'Lizaveta'
], verbose=True)

1 0.0004114379757251594
2 2.8704975050592518e-05
3 1.354699077642759e-06
4 1.531997686279613e-07
5 7.181239154435686e-09
6 1.463567762452925e-10
7 2.6936829983796166e-12
8 1.8635542756085397e-13
9 2.5320030918594287e-15
10 3.0142893950707483e-16


25.76221419471235

In [ ]:
score_bigram_model(model, [
    'Rodion', 'Romanovitch', 'Raskolnikov', 'was',  'a',  'man'
])

26.23989193243241

If n-gram was not in training data, perplexity will be inf

In [ ]:
score_bigram_model(model, ['Rodion', 'Petrovich'])

inf

If use whole text, perplexity will be inf, because the numbers will be too small

In [ ]:
len(text)

209021

In [ ]:
text[:3]

['CRIME', 'AND', 'PUNISHMENT']

In [ ]:
score_bigram_model(model, text[:190])

44.24570450270989

In [ ]:
score_bigram_model(model, text, verbose=True)

1 9.568325016864173e-06
2 9.568325016864173e-06
3 4.7841625084320866e-06
4 1.993401045180036e-07
5 1.993401045180036e-07
6 1.993401045180036e-08
7 1.993401045180036e-08
8 8.30583768825015e-10
9 8.30583768825015e-10
10 8.30583768825015e-10
11 6.229378266187613e-10
12 1.5573445665469032e-10
13 1.5573445665469032e-10
14 4.8414856991095434e-12
15 6.161890889775782e-13
16 1.2837272687032879e-14
17 2.4545454468514106e-17
18 2.4545454468514106e-18
19 7.219251314268855e-21
20 3.779712729983694e-23
21 9.218811536545595e-25
22 6.326387274598953e-28
23 1.2652774549197907e-28
24 1.2652774549197907e-28
25 4.025775170061098e-31
26 2.3004429543206274e-33
27 1.149646653833397e-35
28 1.6192206392019677e-37
29 3.2384412784039357e-38
30 1.040184564797838e-39
31 1.4276483184159181e-43
32 1.5027877035957032e-44
33 7.869259682200317e-46
34 2.1383857832066077e-48
35 5.34596445801652e-50
36 4.6486647461013216e-52
37 2.3243323730506608e-52
38 9.862796491021192e-55
39 6.679094689179136e-57
40 2.903954212686581e

inf

Another formula of perplexity:

$PP(T) = exp\{\frac{1}{N}\sum_{i=1}^N -\log p(w_i|w_1, ..., w_{i-1})\}$



Problems with length of the context (GPT-2 has fixed length = 1024, if $N>1024$, impossible to compute).


Idea: split into chunks

First approach: severeral non-overlapping chunks

Fast, but less context for a lot of steps

Better approach: sliding window of context

Better approximation, but significantly slower

Possible to decrease number of steps by using `stride`, compute perplexity not for every token, but with step $>1$

Example (`stride` = 3,  size of chunk = 5): Lorem ipsum dolor sit amet consectetur adipiscing elit sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. -- (Lorem ipsum dolor sit amet), (sit amet consectetur adipiscing elit), (adipiscing elit sed do eiusmod) ...



Let's compute perplexy for some  model from hugging face:

In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

model_id = 'openai-community/gpt2-medium' #YOUR CODE HERE
model = GPT2LMHeadModel.from_pretrained(model_id)
tokenizer = GPT2TokenizerFast.from_pretrained(model_id)

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
encodings = tokenizer("\n\n".join(text[:1000]), return_tensors="pt")

Token indices sequence length is longer than the specified maximum sequence length for this model (3350 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
encodings['input_ids'].shape

torch.Size([1, 3350])

In [ ]:
max_length = model.config.n_positions
max_length

1024

In [ ]:
#Big to compute faster
stride = 1000

In [ ]:
import torch
import tqdm

seq_len = encodings.input_ids.size(1)
nlls = []
prev_end_loc = 0


for begin_loc in tqdm.tqdm(range(0, seq_len, stride)):


    end_loc = min(begin_loc + max_length, seq_len)
    trg_len = end_loc - prev_end_loc
    input_ids = encodings.input_ids[:, begin_loc:end_loc]
    target_ids = input_ids.clone()
    #Ignore loss not for last token
    target_ids[:, :-trg_len] = -100

    with torch.no_grad():
        outputs = model(input_ids, labels=target_ids)

        #CrossEntropyLoss == -log p(w_i|...)
        neg_log_likelihood = outputs.loss

    nlls.append(neg_log_likelihood)

    prev_end_loc = end_loc
    if end_loc == seq_len:
        break

#Mean+exp
ppl = torch.exp(torch.stack(nlls).mean())

 75%|███████▌  | 3/4 [01:08<00:22, 22.73s/it]


In [ ]:
nlls

[tensor(1.6980), tensor(1.6148), tensor(1.5265), tensor(1.4975)]

In [ ]:
ppl

tensor(4.8753)

#BLEU

In [ ]:
import evaluate

bleu = evaluate.load("bleu")

Unigrams:

In [ ]:
predictions = ["language models"]
references = [["models language"]]


results = bleu.compute(predictions=predictions,
                       references=references,
                       max_order=1)
print(results)

{'bleu': 1.0, 'precisions': [1.0], 'brevity_penalty': 1.0, 'length_ratio': 1.0, 'translation_length': 2, 'reference_length': 2}


Bigrams:

In [ ]:
results = bleu.compute(predictions=predictions,
                       references=references,
                       max_order=2)
print(results)

{'bleu': 0.0, 'precisions': [1.0, 0.0], 'brevity_penalty': 1.0, 'length_ratio': 1.0, 'translation_length': 2, 'reference_length': 2}


In [ ]:
results = bleu.compute(predictions=predictions,
                       references=references,
                       max_order=3)
print(results)

{'bleu': 0.0, 'precisions': [1.0, 0.0, 0.0], 'brevity_penalty': 1.0, 'length_ratio': 1.0, 'translation_length': 2, 'reference_length': 2}


More examples:

In [ ]:
predictions = ["I really loved reading hunger games"]
references = ["I liked reading hunger games"]

bleu.compute(predictions=predictions,
                       references=references,
                       max_order=3)

{'bleu': 0.40548013303822666,
 'precisions': [0.6666666666666666, 0.4, 0.25],
 'brevity_penalty': 1.0,
 'length_ratio': 1.2,
 'translation_length': 6,
 'reference_length': 5}

In [ ]:
BP = min(len(predictions[0].split()) / len(references[0].split()), 1)
BP

1

$Precision = \frac{\# n-gramm\_matches}{\# n-gramm\_in\_prediction}$

In [ ]:
precision_1 = (len(set(predictions[0].split()) & set(references[0].split()))
                / len(set(predictions[0].split())))
precision_1

0.6666666666666666

In [ ]:
from nltk import ngrams

precision_list = []

for i in range(1, 4):
  print(i, end=' ')
  precision_list.append(len(set(ngrams(predictions[0].split(), i)) & set(ngrams(references[0].split(), i)))
                / len(set(ngrams(predictions[0].split(), i))))
  print(precision_list[-1])

1 0.6666666666666666
2 0.4
3 0.25


 BLEU

In [ ]:
BP * np.exp(np.log(precision_list).mean())

0.40548013303822666

## ROUGE

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=e6398643432b574cfc6be59170c038a2fb6da005b8caeca0fdee65c1eb3f1132
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [ ]:
rouge = evaluate.load('rouge')

In [ ]:
predictions = ["language models"]
references = [["models language"]]

rouge.compute(predictions=predictions,
              references=references,
              tokenizer=lambda x: x.split())

{'rouge1': 1.0, 'rouge2': 0.0, 'rougeL': 0.5, 'rougeLsum': 0.5}

In [ ]:
predictions = ["I really loved reading hunger games"]
references = ["I liked reading hunger games"]

In [ ]:
rouge.compute(predictions=predictions,
              references=references,
              tokenizer=lambda x: x.split())

{'rouge1': 0.7272727272727272,
 'rouge2': 0.4444444444444445,
 'rougeL': 0.7272727272727272,
 'rougeLsum': 0.7272727272727272}

ROUGE-1

In [ ]:
rouge1_recall = (len(set(predictions[0].split()) & set(references[0].split()))
                / len(set(references[0].split())))
rouge1_recall

0.8

In [ ]:
rouge1_precision = (len(set(predictions[0].split()) & set(references[0].split()))
                / len(set(predictions[0].split())))
rouge1_precision

0.6666666666666666

In [ ]:
rouge1_F1 = 2 * rouge1_recall * rouge1_precision / (rouge1_recall + rouge1_precision)
rouge1_F1

0.7272727272727272

ROUGE-2 (bigrams)

In [ ]:
from nltk import ngrams

rouge2_recall = (len(set(ngrams(predictions[0].split(), 2)) & set(ngrams(references[0].split(), 2)))
                / len(set(ngrams(references[0].split(), 2))))
rouge2_recall

0.5

In [ ]:
rouge2_precision = (len(set(ngrams(predictions[0].split(), 2)) & set(ngrams(references[0].split(), 2)))
                / len(set(ngrams(predictions[0].split(), 2))))
rouge2_precision

0.4

In [ ]:
rouge2_F1 = 2 * rouge2_recall * rouge2_precision / (rouge2_recall + rouge2_precision)
rouge2_F1

0.4444444444444445

In [ ]:
rouge.compute(predictions=predictions,
              references=references,
              tokenizer=lambda x: x.split())

{'rouge1': 0.7272727272727272,
 'rouge2': 0.4444444444444445,
 'rougeL': 0.7272727272727272,
 'rougeLsum': 0.7272727272727272}

# Bertscore

In [ ]:
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [ ]:
bertscore = evaluate.load("bertscore")
predictions = ["I really loved reading hunger games"]
references = ["I loved reading hunger games"]
results = bertscore.compute(predictions=predictions,
                            references=references,
                            lang="en")

results

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'precision': [0.980343759059906],
 'recall': [0.9934144616127014],
 'f1': [0.9868358373641968],
 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.48.3)'}

In [ ]:
predictions = ["I really loved reading hunger games"]
references = ["2 4 86 23 3 пицца !!!!!"]
results = bertscore.compute(predictions=predictions, references=references, lang="en")

results

{'precision': [0.8170374631881714],
 'recall': [0.735816478729248],
 'f1': [0.7743028402328491],
 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.48.3)'}

In [ ]:
2 * 0.8170374631881714 * 0.735816478729248 / (0.8170374631881714 + 0.735816478729248)

0.7743028663863469

##G-eval

In terms of API Key, we do not have api key

In [ ]:
%env OPENAI_API_KEY=<...your api key here...>

In [ ]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval

test_case = LLMTestCase(input="Say hello",
                        actual_output="Hello everyone")

coherence_metric = GEval(
    name="Coherence",
    criteria="Coherence - the collective quality of all sentences in the actual output",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

coherence_metric.measure(test_case)
print(coherence_metric.score)
print(coherence_metric.reason)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

## Sources:


*   https://huggingface.co/docs/transformers/perplexity
*   https://huggingface.co/spaces/evaluate-metric/rouge
*  https://huggingface.co/spaces/evaluate-metric/bertscore

